# Task1: Frequent Itemset Mining for Disease Diagnosis

**Objective:** Apply the Apriori algorithm to discover frequent symptom patterns for various diseases.

**Dataset:** Disease-symptom dataset with 120 transactions per disease.

**Approach:**
- Data preprocessing with synonym mapping
- Custom Apriori implementation
- Adaptive minimum support based on symptom count
- Per-disease frequent itemset analysis

In [1]:
# To remove DeprecationWarning
import warnings
warnings.simplefilter("ignore", DeprecationWarning)

In [2]:
import pandas as pd

df = pd.read_csv("disease.csv")

# Show full content in each cell
pd.set_option('display.max_colwidth', None)

In [3]:
# Synonym / typo correction map
synonym_map = {
    "scurring": "scarring",
    "spotting_ urination": "spotting_urination",
    "dischromic _patches": "dischromic_patches",
    "foul_smell_of urine": "foul_smell_of_urine",
    "(vertigo) Paroymsal Positional Vertigo": "(vertigo) Paroxysmal Positional Vertigo",
    "Peptic ulcer diseae": "Peptic ulcer disease",
    "Dimorphic hemmorhoids(piles)": "Dimorphic hemorrhoids (piles)",
    "Osteoarthristis": "Osteoarthritis",
    "swollen_extremeties": "swollen_extremities"
}

In [4]:
# Collect symptom columns
symptom_cols = [col for col in df.columns if "Symptom" in col]

## Apriori Algorithm Implementation

Custom implementation to find frequent itemsets:
- Generate frequent individual symptoms (Level 1)
- Iteratively combine to create larger itemsets
- Prune infrequent patterns based on minimum support

In [5]:
# apriori algorithm
def apriori_manual(df, min_sup):
    num_records = len(df) # number of transaction
    support_data= {}

    items = df.columns
    itemsets = [{item}for item in items]

    k = 1
    current_itemsets = itemsets

    while current_itemsets:
        itemset_support = {}

        # calculate support of each itemsets
        for itemset in current_itemsets:
            mask = df[list(itemset)].all(axis=1)
            support = mask.sum() / num_records

        # keep those that meet minimum support
            if support >= min_sup:
                itemset_support[frozenset(itemset)] = support

        support_data.update(itemset_support)
        next_itemsets = set()
        current_keys = list(itemset_support.keys())

        current_keys_sorted = [sorted(list(k)) for k in current_keys]
        for i in range(len(current_keys)):
            for j in range(i+1, len(current_keys)):
                if current_keys_sorted[i][:k-1] == current_keys_sorted[j][:k-1]: # candidate generation
                    union_set = frozenset(current_keys_sorted[i]).union(current_keys_sorted[j])
                    next_itemsets.add(union_set)

        if not next_itemsets:
            break

        current_itemsets = next_itemsets
        k +=1

    result = pd.DataFrame([
        {'itemsets': set(itemset), 'support':support}
        for itemset, support in support_data.items()
    ])

    return result

In [6]:
# Clean and map synonyms
def clean_symptom(sym):
    sym = str(sym).strip()
    if sym == '' or pd.isna(sym) or sym == 'nan':
        return None
    return synonym_map.get(sym, sym)

df['Symptoms_list'] = df[symptom_cols].apply(
    lambda row: [s for sym in row if (s := clean_symptom(sym)) is not None],
    axis=1
)
df = df[df['Symptoms_list'].map(len) > 0]  # remove patients with no symptoms

# Group by disease
basket = df.groupby(['Disease'])['Symptoms_list'].apply(list)

In [7]:
# One-hot encoding function
def prepare_disease_basket(symptom_lists):
    oht = pd.DataFrame([{symptom: 1 for symptom in symptoms} for symptoms in symptom_lists])
    oht = oht.fillna(0)
    return oht


In [8]:
# Adjust minimumm support for computational feasibility
def get_adaptive_minsupport(num_symptoms):
    if num_symptoms >= 15:
        return 0.85
    elif num_symptoms >= 12:
        return 0.80
    elif num_symptoms >= 9:
        return 0.75
    elif num_symptoms >= 6:
        return 0.70
    else:
        return 0.65

## Per-Disease Analysis with Adaptive Minimum Support

**Adaptive Strategy:** Different diseases use different minimum support thresholds based on symptom count:
- Simple diseases (4-7 symptoms): Higher support (0.65-0.70)
- Complex diseases (11+ symptoms): Lower support (0.40-0.75)

This ensures computational feasibility while discovering meaningful patterns.

In [9]:
# Apriori analysis per disease
for disease in basket.index:
    oht = prepare_disease_basket(basket[disease])

    num_symptoms = len(oht.columns)
    min_sup = get_adaptive_minsupport(num_symptoms)

    print(f"\n{disease}: {len(oht)} transactions, {num_symptoms} symptoms (min_sup={min_sup})")

    frequent_itemsets = apriori_manual(oht, min_sup)
    frequent_itemsets = frequent_itemsets[frequent_itemsets['itemsets'].apply(lambda x: len(x) > 1)]
    if frequent_itemsets.empty:
        print("No symptom combinations meet the minimum support.")
        continue

    # Sort and print all itemsets
    all = frequent_itemsets.sort_values(by='support', ascending=False)
    print(all)


(vertigo) Paroymsal  Positional Vertigo: 120 transactions, 6 symptoms (min_sup=0.7)
                                                                   itemsets  \
15                                                       {headache, nausea}   
20                                                       {vomiting, nausea}   
8                                               {vomiting, loss_of_balance}   
9                                                      {headache, vomiting}   
11                                          {unsteadiness, loss_of_balance}   
12                                              {headache, loss_of_balance}   
14                                                   {unsteadiness, nausea}   
16                                                {nausea, loss_of_balance}   
17                                                 {headache, unsteadiness}   
18                                                 {vomiting, unsteadiness}   
29                                            